Set up up the Spark master
  - "local" for local execution
  - "local[*]" for local execution using all core
  - "spark://spark-master:7077" to connect to the Spark master running on the docker container

For the first two options you will to set up a Python environment and install pyspark.

In [ ]:
master="local"

if master == "spark://spark-master:7077":
    filename = '/data/lab01/flights.csv'
else:
    filename = '../data/flights.csv'

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
             .master(master) \
             .appName('flights') \
             .getOrCreate()

df = spark.read.option("delimiter", ",").option("header", True).option("inferSchema", "true").csv(filename)
df.printSchema()
df.show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 18:12:33 WARN Utils: Your hostname, RVs-Desktop, resolves to a loopback address: 127.0.1.1; using 192.168.0.11 instead (on interface wlp35s0)
26/09/16 18:12:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 18:12:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- day of Month: integer (nullable = true)
 |-- day of Week: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- tailnum: string (nullable = true)
 |-- flnum: integer (nullable = true)
 |-- org_id: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest_id: integer (nullable = true)
 |-- dest: string (nullable = true)
 |-- scheduled dep time: integer (nullable = true)
 |-- dep time: integer (nullable = true)
 |-- departure delay: integer (nullable = true)
 |-- scheduled arr time: integer (nullable = true)
 |-- arr time: integer (nullable = true)
 |-- arrival delay: integer (nullable = true)
 |-- elapsed time: integer (nullable = true)
 |-- distance: integer (nullable = true)

+------------+-----------+-------+-------+-----+------+------+-------+----+------------------+--------+---------------+------------------+--------+-------------+------------+--------+
|day of Month|day of Week|carrier|tailnum|flnum|org_id|origin|dest_id|dest|scheduled dep

Flights per airport

In [3]:
result = df.groupBy("origin").count()
result.show()

+------+-----+
|origin|count|
+------+-----+
|   PSE|   67|
|   INL|   50|
|   MSY| 3130|
|   PPG|   10|
|   GEG|  699|
|   SNA| 3159|
|   BUR| 1683|
|   GTF|  149|
|   GRB|  255|
|   IDA|  211|
|   GRR|  720|
|   JLN|   61|
|   PSG|   55|
|   EUG|  416|
|   PVD|  813|
|   GSO|  530|
|   MYR|  106|
|   OAK| 3319|
|   FAR|  368|
|   MSN|  696|
+------+-----+
only showing top 20 rows


In [4]:
ordered = result.orderBy("count", ascending=False)
ordered.show()

+------+-----+
|origin|count|
+------+-----+
|   ATL|28420|
|   DFW|22771|
|   LAX|18051|
|   ORD|17960|
|   DEN|17293|
|   IAH|13466|
|   SFO|13141|
|   PHX|13135|
|   LAS|10835|
|   CLT| 9378|
|   MCO| 8830|
|   SLC| 8579|
|   EWR| 8239|
|   MSP| 7760|
|   SEA| 7663|
|   BOS| 7589|
|   LGA| 7521|
|   MIA| 7212|
|   JFK| 7135|
|   DTW| 6951|
+------+-----+
only showing top 20 rows


The number of flights per route

In [5]:
fpr = df.groupBy("Origin", "Dest").count()
fpr.show()

+------+----+-----+
|Origin|Dest|count|
+------+----+-----+
|   ORD| PDX|  180|
|   SJC| LIH|   18|
|   BQN| MCO|   30|
|   ATL| GSP|  165|
|   SNA| PHX|  352|
|   PHL| MCO|  380|
|   PBI| DCA|   45|
|   EWR| STT|   31|
|   LAS| LIT|   31|
|   MCI| MKE|   54|
|   MDW| MEM|   54|
|   SMF| BUR|  188|
|   MCI| IAH|  189|
|   SPI| ORD|   75|
|   CLE| SJU|    5|
|   DSM| EWR|   17|
|   ROC| CLE|   16|
|   FSD| ATL|   25|
|   LBB| DEN|   55|
|   JFK| ORD|   84|
+------+----+-----+
only showing top 20 rows


 The route with the highest number of flights

In [6]:
ofpr = fpr.orderBy("count", ascending=False)
ofpr.show(5)

+------+----+-----+
|Origin|Dest|count|
+------+----+-----+
|   SFO| LAX| 1080|
|   LAX| SFO| 1074|
|   LAS| LAX| 1025|
|   LAX| LAS| 1021|
|   JFK| LAX|  868|
+------+----+-----+
only showing top 5 rows


The average duration of flights per route

In [26]:
from pyspark.sql.functions import col, avg, round

df2 = df.withColumn("FlightTime",  col("arr time") - col("dep time")).select("origin", "dest", "FlightTime")
avg = df2.groupBy("origin", "dest").agg(round(avg("FlightTime"), 2).alias("AVG")).sort(col("AVG").desc())
avg.show()

+------+----+-------+
|origin|dest|    AVG|
+------+----+-------+
|   GUM| HNL|1086.06|
|   SEA| FLL| 872.32|
|   PDX| BOS| 814.88|
|   SFO| DCA| 797.07|
|   SFO| RDU| 789.35|
|   PDX| IAD| 776.52|
|   PDX| DCA| 776.23|
|   LAX| DCA| 776.03|
|   LAX| PIT| 770.53|
|   SMF| IAD| 762.81|
|   SEA| DCA| 754.43|
|   LIH| SAN| 749.21|
|   SAN| MCO| 747.36|
|   KOA| PDX| 744.39|
|   SAN| BWI| 740.36|
|   OGG| BLI| 735.13|
|   LIH| PDX| 731.29|
|   LAS| IAD| 725.02|
|   LAS| DCA| 724.55|
|   MSP| SJU| 724.25|
+------+----+-------+
only showing top 20 rows
